# Staccato & Scale Marketing Dataset Preparation

This notebook prepares the public bank marketing dataset for Tableau-based enrollment elasticity and seasonal marketing analysis.

In [1]:
import pandas as pd
import numpy as np

## Load the dataset

In [2]:
from google.colab import files
uploaded = files.upload()

Saving bank-additional-full.csv to bank-additional-full.csv


In [3]:
df = pd.read_csv("bank-additional-full.csv", sep=";")
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## Check shape and columns

In [4]:
print(df.shape)
print(df.columns.tolist())


(41188, 21)
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']


## Keep the columns needed for the project

In [5]:
columns_needed = [
    "age",
    "job",
    "education",
    "contact",
    "month",
    "day_of_week",
    "duration",
    "campaign",
    "pdays",
    "previous",
    "poutcome",
    "y"
]

df = df[columns_needed].copy()
df.head()

,age,job,education,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,y
0,56,housemaid,basic.4y,telephone,may,mon,261,1,999,0,nonexistent,no
1,57,services,high.school,telephone,may,mon,149,1,999,0,nonexistent,no
2,37,services,high.school,telephone,may,mon,226,1,999,0,nonexistent,no
3,40,admin.,basic.6y,telephone,may,mon,151,1,999,0,nonexistent,no
4,56,services,high.school,telephone,may,mon,307,1,999,0,nonexistent,no


## Create academy-style mapped fields

In [6]:
# Enrollment outcome
df["Enrollment_Status"] = df["y"].map({"yes": "Enrolled", "no": "Not Enrolled"})

# Promotion type from contact channel
df["Promotion_Type"] = df["contact"].map({
    "cellular": "Digital Campaign",
    "telephone": "Phone Outreach"
})

# Simulated course category from job profile
def map_course_category(job):
    job = str(job).lower()
    if job in ["student", "technician", "admin."]:
        return "Strings"
    elif job in ["management", "entrepreneur", "self-employed"]:
        return "Piano"
    elif job in ["services", "blue-collar", "housemaid"]:
        return "Percussion"
    elif job in ["retired", "unemployed"]:
        return "Voice"
    else:
        return "General Music"

df["Course_Category"] = df["job"].apply(map_course_category)

# Simulated time-to-enrollment using campaign pressure and prior delay fields
df["Time_to_Enrollment"] = np.where(
    df["pdays"] == 999,
    df["campaign"] * 3,
    np.maximum(1, df["pdays"])
)

# Simulated discount rate based on contact intensity
df["Discount_Rate"] = np.where(
    df["campaign"] <= 1, 0.05,
    np.where(df["campaign"] <= 3, 0.10, 0.15)
)

# Estimated LTV based on age and enrollment behavior
df["Estimated_LTV"] = np.where(
    df["Enrollment_Status"] == "Enrolled",
    800 + (df["age"] * 6) + (df["previous"] * 25),
    300 + (df["age"] * 2)
)

df.head()

,age,job,education,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,y,Enrollment_Status,Promotion_Type,Course_Category,Time_to_Enrollment,Discount_Rate,Estimated_LTV
0,56,housemaid,basic.4y,telephone,may,mon,261,1,999,0,nonexistent,no,Not Enrolled,Phone Outreach,Percussion,3,0.05,412
1,57,services,high.school,telephone,may,mon,149,1,999,0,nonexistent,no,Not Enrolled,Phone Outreach,Percussion,3,0.05,414
2,37,services,high.school,telephone,may,mon,226,1,999,0,nonexistent,no,Not Enrolled,Phone Outreach,Percussion,3,0.05,374
3,40,admin.,basic.6y,telephone,may,mon,151,1,999,0,nonexistent,no,Not Enrolled,Phone Outreach,Strings,3,0.05,380
4,56,services,high.school,telephone,may,mon,307,1,999,0,nonexistent,no,Not Enrolled,Phone Outreach,Percussion,3,0.05,412


## Optional cleanup for Tableau

In [7]:
df["month"] = df["month"].str.title()
df["day_of_week"] = df["day_of_week"].str.title()
df["Course_Category"] = df["Course_Category"].astype(str)
df["Promotion_Type"] = df["Promotion_Type"].astype(str)

## Save the Tableau-ready file

In [8]:
df.to_csv("staccato_scale_marketing_ready.csv", index=False)
print("Saved as staccato_scale_marketing_ready.csv")

Saved as staccato_scale_marketing_ready.csv


In [9]:
from google.colab import files
files.download("staccato_scale_marketing_ready.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>